# 01 Make shared dataset splits

**AI-Assisted Mineral Identification — Capstone, Fall 2026 (SO-1, SO-2)**

Produces the split manifest for all preliminary tests, so all nine experiments use exactly the same train / validation / test images.

What it does:
1. Finds the 15 selected mineral classes in the dataset folder on Google Drive and assigns each to its priority group.
2. Skips unreadable files.
3. Hashes every image: SHA-256 for exact duplicates, perceptual hash for near-duplicates (same photo resized or recompressed).
4. Groups near-duplicates into clusters so a cluster never ends up split between train and test.
5. Splits each class individually 70 / 15 / 15, keeping clusters together.
6. Verifies that no file and no near-duplicate cluster appears in more than one split.
7. Reports aspect ratios per class (for letterbox padding).
8. Writes `splits.csv`, `splits_meta.json` and `excluded.csv`.

The split is done once over all 15 classes; each 5-class group is just a filter on the `group` column.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install imagehash

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 4.7 MB/s eta 0:00:00


In [ ]:
import os, io, re, json, hashlib, platform, datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import imagehash
from tqdm.auto import tqdm
from IPython.display import display

## 2. Configuration

In [ ]:
# ---- Drive Paths ----
DATA_DIR = "/content/drive/MyDrive/CSE_Capstone_Data/mineral-photos"
OUT_DIR  = "/content/drive/MyDrive/CSE_Capstone_Data/splits"

# ---- Shared split settings ----
SEED = 42
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
PHASH_THRESHOLD = 6    # max Hamming distance (out of 64 bits) for two images to count as near-duplicates
NUM_WORKERS = 16       # parallel file reads

GROUPS = {
    "high":   ["baryte", "beryl", "copper", "fluorite", "quartz"],
    "medium": ["azurite", "malachite", "smithsonite", "cerussite", "pyromorphite"],
    "low":    ["wulfenite", "calcite", "gypsum", "hematite", "pyrite"],
}
# Alternate folder spellings that should map to our class names
ALIASES = {"barite": "baryte", "haematite": "hematite", "native copper": "copper"}

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
GROUP_ORDER = list(GROUPS)
CLASS_TO_GROUP = {c: g for g, cs in GROUPS.items() for c in cs}
assert len(CLASS_TO_GROUP) == 15
assert abs(sum(RATIOS.values()) - 1) < 1e-9
assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Find the 15 classes

Each image's class is the name of the folder it sits in. Folder names are matched case-insensitively, with `_` and `-` treated as spaces. If a class is missing, the cell stops and names it. Add it to the dataset's spelling to `ALIASES`.

In [ ]:
def norm(name):
    n = re.sub(r"[\s_\-]+", " ", name.strip().lower())
    return ALIASES.get(n, n)

rows, folder_counts, folder_class = [], Counter(), {}
for root, dirs, files in os.walk(DATA_DIR):
    dirs.sort()                                   # deterministic walk order
    imgs = sorted(f for f in files if os.path.splitext(f)[1].lower() in IMG_EXT)
    if not imgs:
        continue
    rel_folder = os.path.relpath(root, DATA_DIR)
    folder_counts[rel_folder] = len(imgs)
    cls = norm(os.path.basename(root))
    if cls in CLASS_TO_GROUP:
        folder_class[rel_folder] = cls
        rows += [{"rel_path": os.path.join(rel_folder, f), "class": cls} for f in imgs]

print(f"Found {len(folder_counts)} image folders, {sum(folder_counts.values())} images in total.\n")
print("Matched folders:")
for f, c in sorted(folder_class.items(), key=lambda kv: (GROUP_ORDER.index(CLASS_TO_GROUP[kv[1]]), kv[1])):
    print(f"  [{CLASS_TO_GROUP[c]:6}] {c:13} <- {f}  ({folder_counts[f]} images)")

unmatched = sorted(set(folder_counts) - set(folder_class))
print(f"\nUnmatched folders (ignored): {len(unmatched)}")
if unmatched:
    print("  " + ", ".join(os.path.basename(f) for f in unmatched[:60]) + (" ..." if len(unmatched) > 60 else ""))

missing = sorted(set(CLASS_TO_GROUP) - set(folder_class.values()))
assert not missing, f"No folder found for: {missing}. Add the dataset's spelling to ALIASES."
merged = [c for c, n in Counter(folder_class.values()).items() if n > 1]
if merged:
    print("\nNOTE: several folders map to", merged, "- their images are combined.")

df = pd.DataFrame(rows)
print(f"\n{len(df)} images belong to the 15 selected classes.")

Found 29 image folders, 57000 images in total.

Matched folders:
  [high  ] baryte        <- data/data/baryte  (2211 images)
  [high  ] baryte        <- mineral-trier/baryte  (1083 images)
  [high  ] beryl         <- data/data/beryl  (1615 images)
  [high  ] beryl         <- mineral-trier/beryl  (829 images)
  [high  ] copper        <- data/data/copper  (1429 images)
  [high  ] copper        <- mineral-trier/copper  (722 images)
  [high  ] fluorite      <- data/data/fluorite  (5669 images)
  [high  ] quartz        <- data/data/quartz  (8244 images)
  [high  ] quartz        <- mineral-trier/quartz  (5163 images)
  [medium] azurite       <- data/data/azurite  (1250 images)
  [medium] azurite       <- mineral-trier/azurite  (623 images)
  [medium] cerussite     <- data/data/cerussite  (1367 images)
  [medium] cerussite     <- mineral-trier/cerussite  (519 images)
  [medium] malachite     <- data/data/malachite  (1607 images)
  [medium] malachite     <- mineral-trier/malachite  (652 images

## 4. Read and hash every image

Each file is opened fully (so truncated or corrupt files are caught), rotated according to its camera EXIF tag, and hashed.

In [ ]:
def inspect(rel_path):
    try:
        with open(os.path.join(DATA_DIR, rel_path), "rb") as fh:
            data = fh.read()
        with Image.open(io.BytesIO(data)) as im:
            im = ImageOps.exif_transpose(im)   # apply camera rotation, as training will
            im = im.convert("RGB")             # forces a full decode
            w, h = im.size
            ph = str(imagehash.phash(im))
        return {"sha256": hashlib.sha256(data).hexdigest(), "phash": ph, "width": w, "height": h, "error": ""}
    except Exception as e:
        return {"sha256": "", "phash": "", "width": 0, "height": 0, "error": f"{type(e).__name__}: {e}"}

with ThreadPoolExecutor(NUM_WORKERS) as ex:
    info = list(tqdm(ex.map(inspect, df.rel_path), total=len(df), desc="Reading + hashing"))
df = pd.concat([df, pd.DataFrame(info)], axis=1)

bad = df[df.error != ""]
df = df[df.error == ""].reset_index(drop=True)
print(f"Unreadable files skipped: {len(bad)}")
for _, r in bad.head(10).iterrows():
    print(f"  {r.rel_path}: {r.error}")

# Exact copies inside the same class add nothing but extra weight: keep the first one.
dupe_mask = df.duplicated(subset=["class", "sha256"], keep="first")
exact_dupes = df[dupe_mask]
df = df[~dupe_mask].reset_index(drop=True)
print(f"Exact duplicates within the same class removed: {len(exact_dupes)}")

excluded = pd.concat([bad.assign(reason="unreadable"),
                      exact_dupes.assign(reason="exact duplicate within class")], ignore_index=True)
print(f"\n{len(df)} images remain.")

Reading + hashing:   0%|          | 0/57000 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 12 bytes but only got 2. 
  warnings.warn(str(msg))
/usr/local/lib/python3.13/dist-packages/PIL/JpegImagePlugin.py:887: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


Unreadable files skipped: 5
  data/data/cerussite/image_0064882.jpeg: OSError: image file is truncated (4 bytes not processed)
  data/data/cerussite/image_0064883.jpeg: OSError: image file is truncated (14 bytes not processed)
  data/data/fluorite/image_0064902.jpeg: OSError: image file is truncated (33 bytes not processed)
  data/data/quartz/image_0014124.jpeg: OSError: image file is truncated (72 bytes not processed)
  data/data/smithsonite/image_0014147.jpeg: OSError: image file is truncated (47 bytes not processed)
Exact duplicates within the same class removed: 17646

39349 images remain.


## Checkpoint
Saves the hashed files in case the session closes.

In [ ]:
df.to_pickle(os.path.join(OUT_DIR, "_hash_cache.pkl"))
exact_dupes.to_pickle(os.path.join(OUT_DIR, "_dupes_cache.pkl"))
excluded.to_pickle(os.path.join(OUT_DIR, "_excluded_cache.pkl"))

## 5. Group near-duplicates

Exact copies across different classes are also caught here (distance 0). Two images are near-duplicates when their perceptual hashes differ in at most `PHASH_THRESHOLD` of 64 bits. Near-duplicates are linked into clusters (if A≈B and B≈C, all three form one cluster), and every cluster later goes into a single split.

A perceptual hash catches resized, recompressed or lightly color-adjusted copies of the same photo. Cropped copies may not be detected, and the same specimen photographed from a different angle cannot be detected.

In [ ]:
N = len(df)
bits = np.unpackbits(
    np.frombuffer(b"".join(bytes.fromhex(h) for h in df.phash), dtype=np.uint8).reshape(N, 8), axis=1)
S = bits.astype(np.float32) * 2 - 1          # 0/1 -> -1/+1, so a dot product equals 64 - 2 * Hamming distance

parent = list(range(N))                      # union-find
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

n_pairs, CHUNK = 0, 1024
for i0 in tqdm(range(0, N, CHUNK), desc="Comparing hashes"):
    D = (64 - S[i0:i0 + CHUNK] @ S.T) / 2     # Hamming distances for this block of rows
    ii, jj = np.nonzero(D <= PHASH_THRESHOLD)
    ii = ii + i0
    keep = jj > ii                            # each pair once, skip self-matches
    for a, b in zip(ii[keep].tolist(), jj[keep].tolist()):
        n_pairs += 1
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

df["cluster_id"] = pd.factorize(np.array([find(i) for i in range(N)]))[0]
csize = df.cluster_id.value_counts()
multi = csize[csize > 1]
classes_per_cluster = df.groupby("cluster_id")["class"].nunique()

print(f"Near-duplicate pairs found: {n_pairs}")
print(f"Images in multi-image clusters: {int(multi.sum())} (in {len(multi)} clusters)")
print(f"Largest clusters: {multi.head(5).tolist()}")
print(f"Clusters containing more than one class: {int((classes_per_cluster > 1).sum())}")
if len(multi) and multi.max() > 25:
    print("WARNING: a very large cluster usually means the threshold is too loose "
          "(e.g. different specimens on the same plain background). Lower PHASH_THRESHOLD to 4 and re-run from section 5.")

Comparing hashes:   0%|          | 0/39 [00:00<?, ?it/s]

Near-duplicate pairs found: 609
Images in multi-image clusters: 982 (in 422 clusters)
Largest clusters: [18, 8, 8, 7, 6]
Clusters containing more than one class: 206


## 6. Stratified 70 / 15 / 15 split, per class

For each class separately, its clusters are shuffled with the fixed seed and laid out in order; the first 15% go to test, the next 15% to validation and the rest to training. Every class therefore keeps its own 70/15/15 proportion.

A cluster that spans two classes (the same photo filed under two minerals) belongs to its most common class, and all its images get that cluster's split, so the photo can never be in train and test at once.

In [ ]:
rng = np.random.default_rng(SEED)
primary = df.groupby("cluster_id")["class"].agg(lambda s: sorted(s.mode())[0])
cut_test = RATIOS["test"]
cut_val = RATIOS["test"] + RATIOS["val"]

cluster_split = {}
for cls in sorted(CLASS_TO_GROUP):
    ids = np.array(sorted(primary[primary == cls].index))
    ids = ids[rng.permutation(len(ids))]
    total = csize[ids].sum()
    done = 0
    for cid in ids:
        mid = (done + csize[cid] / 2) / total   # where the cluster's midpoint falls in this class
        cluster_split[cid] = "test" if mid < cut_test else "val" if mid < cut_val else "train"
        done += csize[cid]

df["split"] = df.cluster_id.map(cluster_split)
df["group"] = pd.Categorical(df["class"].map(CLASS_TO_GROUP), categories=GROUP_ORDER, ordered=True)

## 7. Verify

In [ ]:
SPLITS = ["train", "val", "test"]
assert df.split.notna().all(), "Some images received no split"
assert df.groupby("sha256").split.nunique().max() == 1, "The same file appears in more than one split"
assert df.groupby("cluster_id").split.nunique().max() == 1, "A near-duplicate cluster was split apart"

counts = pd.crosstab([df.group, df["class"]], df.split).reindex(columns=SPLITS, fill_value=0)
pct = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)
table = counts.astype(str) + "  (" + pct.astype(str) + "%)"
table["total"] = counts.sum(axis=1)
display(table)
display(pd.crosstab(df.group, df.split).reindex(columns=SPLITS, fill_value=0).assign(total=lambda t: t.sum(axis=1)))

empty = counts[(counts == 0).any(axis=1)]
assert empty.empty, f"Some classes have an empty split:\n{empty}"
small = counts[counts.min(axis=1) < 10]
if not small.empty:
    print("NOTE: these classes have fewer than 10 images in some split; their per-class metrics will be noisy:")
    print(small)
print("All checks passed: no file and no near-duplicate cluster appears in more than one split.")

split                        train            val           test  total
group  class                                                           
high   baryte        1547  (70.0%)   331  (15.0%)   333  (15.1%)   2211
       beryl         1133  (70.2%)   243  (15.0%)   239  (14.8%)   1615
       copper        1001  (70.0%)   214  (15.0%)   214  (15.0%)   1429
       fluorite      3965  (70.0%)   852  (15.0%)   851  (15.0%)   5668
       quartz        5766  (70.0%)  1240  (15.0%)  1237  (15.0%)   8243
medium azurite        875  (70.0%)   188  (15.0%)   187  (15.0%)   1250
       cerussite      956  (70.0%)   205  (15.0%)   204  (14.9%)   1365
       malachite     1124  (69.9%)   242  (15.1%)   241  (15.0%)   1607
       pyromorphite  1196  (70.1%)   255  (15.0%)   254  (14.9%)   1705
       smithsonite    829  (69.8%)   179  (15.1%)   179  (15.1%)   1187
low    calcite       4605  (70.0%)   980  (14.9%)   989  (15.0%)   6574
       gypsum         947  (70.1%)   201  (14.9%)   203  (15.0%)   1351
       hematite       900  (70.1%)   192  (15.0%)   192  (15.0%)   1284
       pyrite        1639  (70.1%)   350  (15.0%)   350  (15.0%)   2339
       wulfenite     1063  (69.9%)   230  (15.1%)   228  (15.0%)   1521

split,train,val,test,total
group,,,,
high,13412,2880,2874,19166
medium,4980,1069,1065,7114
low,9154,1953,1962,13069


All checks passed: no file and no near-duplicate cluster appears in more than one split.


## 8. Aspect ratios for letterbox padding

Letterboxing adds borders to non-square photos: wide photos get bars on top and bottom, tall photos on the sides. That is harmless **unless the shape of the photos differs by class** — then the border position becomes a clue the model can learn instead of the mineral.

**TO DO**: Read the table to see if one class is mostly *portrait* while the others are mostly *landscape*. The last column shows how many images are smaller than the model input and will be upscaled.

In [ ]:
INPUT_SIZE = 224
df["aspect"] = df.width / df.height
ar = df.groupby("class", observed=True).agg(
    group=("group", "first"),
    median_w_over_h=("aspect", "median"),
    landscape_pct=("aspect", lambda a: (a > 1.2).mean() * 100),
    near_square_pct=("aspect", lambda a: ((a >= 1 / 1.2) & (a <= 1.2)).mean() * 100),
    portrait_pct=("aspect", lambda a: (a < 1 / 1.2).mean() * 100),
)
ar["upscaled_pct"] = df.assign(s=np.minimum(df.width, df.height) < INPUT_SIZE).groupby("class").s.mean() * 100
display(ar.sort_values(["group", "class"]).round(2))

,group,median_w_over_h,landscape_pct,near_square_pct,portrait_pct,upscaled_pct
class,,,,,,
baryte,high,1.33,69.47,23.56,6.97,0.63
beryl,high,1.09,43.96,35.73,20.31,0.56
copper,high,1.25,53.67,28.69,17.63,1.61
fluorite,high,1.26,56.35,38.27,5.38,0.51
quartz,high,1.29,56.81,27.61,15.58,1.21
azurite,medium,1.32,60.48,32.48,7.04,0.72
cerussite,medium,1.33,65.64,23.15,11.21,0.07
malachite,medium,1.33,66.89,26.94,6.16,0.81
pyromorphite,medium,1.33,65.16,26.86,7.98,0.35


## 9. Save

Files written to `OUT_DIR`:

| File | Contents |
|---|---|
| `splits.csv` | One row per image: relative path, group, class, split, cluster, hashes, size. |
| `splits_meta.json` | Seed, ratios, threshold, counts, library versions, and the SHA-256 of `splits.csv`. |
| `excluded.csv` | Unreadable files and removed exact duplicates, with the reason. |

**TO DO**: Commit to GitHub.

In [ ]:
COLS = ["rel_path", "group", "class", "split", "cluster_id", "sha256", "phash", "width", "height"]
out = df.sort_values(["group", "class", "rel_path"])[COLS]
csv_path = os.path.join(OUT_DIR, "splits.csv")
out.to_csv(csv_path, index=False, lineterminator="\n")
with open(csv_path, "rb") as fh:
    manifest_sha = hashlib.sha256(fh.read()).hexdigest()

excluded.reindex(columns=["rel_path", "class", "reason", "error", "sha256"]).to_csv(
    os.path.join(OUT_DIR, "excluded.csv"), index=False, lineterminator="\n")

per_class = df.groupby(["class", "split"]).size().unstack(fill_value=0).reindex(columns=SPLITS, fill_value=0)
meta = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "splits_csv_sha256": manifest_sha,
    "seed": SEED,
    "ratios": RATIOS,
    "phash_threshold": PHASH_THRESHOLD,
    "groups": GROUPS,
    "aliases": ALIASES,
    "n_images": int(len(df)),
    "n_excluded": int(len(excluded)),
    "n_multi_image_clusters": int(len(multi)),
    "counts": {c: {s: int(v) for s, v in row.items()} for c, row in per_class.iterrows()},
    "dataset_folder": os.path.basename(os.path.normpath(DATA_DIR)),
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "pillow": PIL.__version__, "imagehash": imagehash.__version__},
}
with open(os.path.join(OUT_DIR, "splits_meta.json"), "w") as fh:
    json.dump(meta, fh, indent=2)

print(f"Wrote {len(out)} rows to {csv_path}")
print(f"splits.csv SHA-256: {manifest_sha}")

Wrote 39349 rows to /content/drive/MyDrive/CSE_Capstone_Data/splits/splits.csv
splits.csv SHA-256: 24f264149e95f3cc4d442f1394032b60dbf2579e590638a601f3c7bb2fa937ed


## To load the splits

```python
import pandas as pd, os
DATA_DIR = "/content/drive/MyDrive/<your path>/mineral-photos"
splits = pd.read_csv("splits.csv")
group = splits[splits.group == "high"]                 # or "medium", "low"
train = group[group.split == "train"]
paths = [os.path.join(DATA_DIR, p) for p in train.rel_path]
```